<a href="https://colab.research.google.com/github/Farzad-Morshedi/f1-performance-analytics/blob/main/notebooks/01_F1_Initial_Data_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 1: Discovery





# Data Provenance & Acquisition
*   **Data Source:** All datasets were acquired from the **Ergast Motor Sport Database**, a comprehensive historical record of Formula 1 telemetry and results.
*   **Ingestion Process:** Three primary CSV files were loaded into the environment: results.csv, drivers.csv, and races.csv.
*   **Validation** Final row counts (26,759 results, 861 drivers, 1,125 races) match the expected historical population of the sport since 1950.


Import needed Libraries


In [38]:
# import needed libraries for EDA
import pandas as pd
import numpy as np

Load the Dataset


In [39]:
#Loading the primary tables
df_results = pd.read_csv('/content/results.csv')
df_drivers = pd.read_csv('/content/drivers.csv')
df_races = pd.read_csv('/content/races.csv')

Check How many rows of data we have in each table

In [40]:
#checking the shape to see the scale of the data
print(f"Results Rows: {df_results.shape[0]}")
print(f"Drivers Rows: {df_drivers.shape[0]}")
print(f"Races Rows: {df_races.shape[0]}")


Results Rows: 26759
Drivers Rows: 861
Races Rows: 1125


# **Results Table**
Checking the characteristics of the data in the results table


*   Shape
*   Columns
*   Index



In [41]:
# Check the scale (Rows, Columns)
print(f"Results Shape: {results.shape}")

# Check the Name of the columns
print(f"Results Columns: {results.columns}")

# Check the Index
print(f"Results Index: {results.index}")


Results Shape: (26759, 18)
Results Columns: Index(['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid',
       'position', 'positionText', 'positionOrder', 'points', 'laps', 'time',
       'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
       'fastestLapSpeed', 'statusId'],
      dtype='object')
Results Index: RangeIndex(start=0, stop=26759, step=1)


In [42]:
# Taking a quick look at the results dataframe using head()
results.head(10)


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1
5,6,18,6,3,8,13,6,6,6,3.0,57,\N,\N,50,14,1:29.639,212.974,11
6,7,18,7,5,14,17,7,7,7,2.0,55,\N,\N,54,8,1:29.534,213.224,5
7,8,18,8,6,1,15,8,8,8,1.0,53,\N,\N,20,4,1:27.903,217.180,5
8,9,18,9,2,4,2,\N,R,9,0.0,47,\N,\N,15,9,1:28.753,215.100,4
9,10,18,10,7,12,18,\N,R,10,0.0,43,\N,\N,23,13,1:29.558,213.166,3


First thing I notice here is that we have \N and R in some columns that have numeric data.

This will most certainly cause pandas to have issues determining the correct Dtype of the column
*   \N means no value is provided (missing value)
*   R in the positionText column means 'Retired from Race". Does this column need to be converted to numeric? This is something to think about before I begin analysis and eventual modeling.



In [43]:
# Doing a schema check on the results table
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26759 entries, 0 to 26758
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   resultId         26759 non-null  int64  
 1   raceId           26759 non-null  int64  
 2   driverId         26759 non-null  int64  
 3   constructorId    26759 non-null  int64  
 4   number           26759 non-null  object 
 5   grid             26759 non-null  int64  
 6   position         26759 non-null  object 
 7   positionText     26759 non-null  object 
 8   positionOrder    26759 non-null  int64  
 9   points           26759 non-null  float64
 10  laps             26759 non-null  int64  
 11  time             26759 non-null  object 
 12  milliseconds     26759 non-null  object 
 13  fastestLap       26759 non-null  object 
 14  rank             26759 non-null  object 
 15  fastestLapTime   26759 non-null  object 
 16  fastestLapSpeed  26759 non-null  object 
 17  statusId    

# Issues with Dtype of Columns
There are some columns that have object Dtype as I suspected.

The ones that could potentially affect the analysis are:
*   number
*   position
*   positionText
*   time
*   milliseconds
*   fastestLap
*   rank
*   fastestLapTime
*   fastestLapSpeed
*   List item








In [53]:
# Checking the Dtypes
df_results.dtypes

,0
resultId,int64
raceId,int64
driverId,int64
constructorId,int64
number,object
grid,int64
position,object
positionText,object
positionOrder,int64
points,float64


# Problems with the schema
Some of the columns have Dtype of object and need to be converted to appropriate Dtype using .astype()

First I need to check if there is a problem with the way data has been entered.


*   Are there leading spaces
*   I googled and found that \N placeholder could be problematic



In [55]:
# Finding if \N is the culprit
# Create a mask to find the string'\N'
n_string_counts_results = (df_results == r'\N').sum()
n_string_counts_results

,0
resultId,0
raceId,0
driverId,0
constructorId,0
number,6
grid,0
position,10953
positionText,0
positionOrder,0
points,0


# Drivers Table
Checking the characteristics of the data in the Drivers table
*   Shape
*   Columns
*   Index







In [46]:
# Check the scale (Rows, Columns)
print(f"Results Shape: {df_drivers.shape}")

Results Shape: (861, 9)


In [48]:
# Check the Name of the columns
print(f"Results Columns: {df_drivers.columns}")

Results Columns: Index(['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob',
       'nationality', 'url'],
      dtype='object')


In [49]:
# Check the Index
print(f"Results Index: {df_drivers.index}")

Results Index: RangeIndex(start=0, stop=861, step=1)


Taking a quick look at the top 10 rows of the table

In [50]:
df_drivers.head(10)

,driverId,driverRef,number,code,forename,surname,dob,nationality,url
0,1,hamilton,44,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton
1,2,heidfeld,\N,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld
2,3,rosberg,6,ROS,Nico,Rosberg,1985-06-27,German,http://en.wikipedia.org/wiki/Nico_Rosberg
3,4,alonso,14,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso
4,5,kovalainen,\N,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen
5,6,nakajima,\N,NAK,Kazuki,Nakajima,1985-01-11,Japanese,http://en.wikipedia.org/wiki/Kazuki_Nakajima
6,7,bourdais,\N,BOU,Sébastien,Bourdais,1979-02-28,French,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bo...
7,8,raikkonen,7,RAI,Kimi,Räikkönen,1979-10-17,Finnish,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C...
8,9,kubica,88,KUB,Robert,Kubica,1984-12-07,Polish,http://en.wikipedia.org/wiki/Robert_Kubica
9,10,glock,\N,GLO,Timo,Glock,1982-03-18,German,http://en.wikipedia.org/wiki/Timo_Glock


There are again some \N entries in the number column. That is for the drivers number in each race. I suspect that these are for reserve drivers that aren't a permanent part of the team and very likely called up from F2 to cover for one of the main 2 drivers of the team.

That said, Kovalainen and Glock not having driver numbers is unuausal. I need to therefore look at more data and possible import more tables from the data set to complete the analysis such as the following tables:
*   status
*   drivers_standings
*   constructors
*   seasons







Checking info() and Dtypes

In [51]:
df_drivers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 861 entries, 0 to 860
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   driverId     861 non-null    int64 
 1   driverRef    861 non-null    object
 2   number       861 non-null    object
 3   code         861 non-null    object
 4   forename     861 non-null    object
 5   surname      861 non-null    object
 6   dob          861 non-null    object
 7   nationality  861 non-null    object
 8   url          861 non-null    object
dtypes: int64(1), object(8)
memory usage: 60.7+ KB


In [52]:
df_drivers.dtypes

,0
driverId,int64
driverRef,object
number,object
code,object
forename,object
surname,object
dob,object
nationality,object
url,object


In [56]:
# Checking for '\N' counts in the df_drivers
n_string_counts_drivers = (df_drivers == r'\N').sum()
n_string_counts_drivers

,0
driverId,0
driverRef,0
number,802
code,757
forename,0
surname,0
dob,0
nationality,0
url,0


In [66]:
# Finding unique values in the code column
print(f"Unique values in the 'code' column:")
print(df_drivers['code'].unique())

Unique values in the 'code' column:
['HAM' 'HEI' 'ROS' 'ALO' 'KOV' 'NAK' 'BOU' 'RAI' 'KUB' 'GLO' 'SAT' 'PIQ'
 'MAS' 'COU' 'TRU' 'SUT' 'WEB' 'BUT' 'DAV' 'VET' 'FIS' 'BAR' 'SCH' 'LIU'
 'WUR' 'SPE' 'ALB' 'WIN' 'YAM' 'MSC' 'MON' 'KLI' 'TMO' 'IDE' 'VIL' 'FMO'
 'DLR' 'DOO' 'KAR' 'FRI' 'ZON' 'PIZ' '\\N' 'BUE' 'BAD' 'MAG' 'ALG' 'GRO'
 'KOB' 'BIA' 'GAS' 'HUL' 'PET' 'DIG' 'SEN' 'CHA' 'MAL' 'DIR' 'PER' 'DAM'
 'RIC' 'VER' 'PIC' 'CHI' 'GUT' 'BOT' 'VDG' 'KVY' 'LOT' 'ERI' 'STE' 'NAS'
 'SAI' 'MER' 'RSS' 'PAL' 'WEH' 'HAR' 'VAN' 'OCO' 'STR' 'GIO' 'LEC' 'SIR'
 'NOR' 'RUS' 'LAT' 'FIT' 'AIT' 'TSU' 'MAZ' 'ZHO' 'DEV' 'PIA' 'SAR' 'LAW'
 'BEA' 'COL']


In [65]:
# looking at the drivers without a code
df_drivers[df_drivers['code'] == '\\N']

,driverId,driverRef,number,code,forename,surname,dob,nationality,url
42,43,matta,\N,\N,Cristiano,da Matta,1973-09-19,Brazilian,http://en.wikipedia.org/wiki/Cristiano_da_Matta
43,44,panis,\N,\N,Olivier,Panis,1966-09-02,French,http://en.wikipedia.org/wiki/Olivier_Panis
44,45,pantano,\N,\N,Giorgio,Pantano,1979-02-04,Italian,http://en.wikipedia.org/wiki/Giorgio_Pantano
45,46,bruni,\N,\N,Gianmaria,Bruni,1981-05-30,Italian,http://en.wikipedia.org/wiki/Gianmaria_Bruni
46,47,baumgartner,\N,\N,Zsolt,Baumgartner,1981-01-01,Hungarian,http://en.wikipedia.org/wiki/Zsolt_Baumgartner
...,...,...,...,...,...,...,...,...,...
802,802,serafini,\N,\N,Dorino,Serafini,1909-07-22,Italian,http://en.wikipedia.org/wiki/Dorino_Serafini
803,803,cantrell,\N,\N,Bill,Cantrell,1908-01-31,American,http://en.wikipedia.org/wiki/William_Cantrell
804,804,mantz,\N,\N,Johnny,Mantz,1918-09-18,American,http://en.wikipedia.org/wiki/Johnny_Mantz
805,805,kladis,\N,\N,Danny,Kladis,1917-02-10,American,http://en.wikipedia.org/wiki/Danny_Kladis


The drivers above don't have code or number it seems. Lets check to see if they also have no number

In [68]:
#Checking drivers without a number
df_drivers[df_drivers['number'] == '\\N']

,driverId,driverRef,number,code,forename,surname,dob,nationality,url
1,2,heidfeld,\N,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld
4,5,kovalainen,\N,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen
5,6,nakajima,\N,NAK,Kazuki,Nakajima,1985-01-11,Japanese,http://en.wikipedia.org/wiki/Kazuki_Nakajima
6,7,bourdais,\N,BOU,Sébastien,Bourdais,1979-02-28,French,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bo...
9,10,glock,\N,GLO,Timo,Glock,1982-03-18,German,http://en.wikipedia.org/wiki/Timo_Glock
...,...,...,...,...,...,...,...,...,...
810,811,bruno_senna,\N,SEN,Bruno,Senna,1983-10-15,Brazilian,http://en.wikipedia.org/wiki/Bruno_Senna
811,812,chandhok,\N,CHA,Karun,Chandhok,1984-01-19,Indian,http://en.wikipedia.org/wiki/Karun_Chandhok
815,816,ambrosio,\N,DAM,Jérôme,d'Ambrosio,1985-12-27,Belgian,http://en.wikipedia.org/wiki/J%C3%A9r%C3%B4me_...
818,819,pic,\N,PIC,Charles,Pic,1990-02-15,French,http://en.wikipedia.org/wiki/Charles_Pic


In [74]:
# checking drivers without name and code
df_drivers[(df_drivers['number'] == '\\N') & (df_drivers['code'] == '\\N')]


,driverId,driverRef,number,code,forename,surname,dob,nationality,url
42,43,matta,\N,\N,Cristiano,da Matta,1973-09-19,Brazilian,http://en.wikipedia.org/wiki/Cristiano_da_Matta
43,44,panis,\N,\N,Olivier,Panis,1966-09-02,French,http://en.wikipedia.org/wiki/Olivier_Panis
44,45,pantano,\N,\N,Giorgio,Pantano,1979-02-04,Italian,http://en.wikipedia.org/wiki/Giorgio_Pantano
45,46,bruni,\N,\N,Gianmaria,Bruni,1981-05-30,Italian,http://en.wikipedia.org/wiki/Gianmaria_Bruni
46,47,baumgartner,\N,\N,Zsolt,Baumgartner,1981-01-01,Hungarian,http://en.wikipedia.org/wiki/Zsolt_Baumgartner
...,...,...,...,...,...,...,...,...,...
802,802,serafini,\N,\N,Dorino,Serafini,1909-07-22,Italian,http://en.wikipedia.org/wiki/Dorino_Serafini
803,803,cantrell,\N,\N,Bill,Cantrell,1908-01-31,American,http://en.wikipedia.org/wiki/William_Cantrell
804,804,mantz,\N,\N,Johnny,Mantz,1918-09-18,American,http://en.wikipedia.org/wiki/Johnny_Mantz
805,805,kladis,\N,\N,Danny,Kladis,1917-02-10,American,http://en.wikipedia.org/wiki/Danny_Kladis


These enteries can be dropped from the drivers table if needed as temporary drivers are not considered in my analysis

# RacesTable
Checking the characteristics of the data in the Drivers table
*   Shape
*   Columns
*   Index

In [77]:
# Check the scale (Rows, Columns)
print(f"Results Shape: {df_races.shape}")

Results Shape: (1125, 18)


In [78]:
# Check the Name of the columns
print(f"Results Columns: {df_races.columns}")

Results Columns: Index(['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url',
       'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time',
       'quali_date', 'quali_time', 'sprint_date', 'sprint_time'],
      dtype='object')


In [79]:
# Check the Index
print(f"Results Index: {df_races.index}")

Results Index: RangeIndex(start=0, stop=1125, step=1)


In [80]:
df_races.head(10)

,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
0,1,2009,1,1,Australian Grand Prix,2009-03-29,06:00:00,http://en.wikipedia.org/wiki/2009_Australian_G...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
1,2,2009,2,2,Malaysian Grand Prix,2009-04-05,09:00:00,http://en.wikipedia.org/wiki/2009_Malaysian_Gr...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
2,3,2009,3,17,Chinese Grand Prix,2009-04-19,07:00:00,http://en.wikipedia.org/wiki/2009_Chinese_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
3,4,2009,4,3,Bahrain Grand Prix,2009-04-26,12:00:00,http://en.wikipedia.org/wiki/2009_Bahrain_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
4,5,2009,5,4,Spanish Grand Prix,2009-05-10,12:00:00,http://en.wikipedia.org/wiki/2009_Spanish_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
5,6,2009,6,6,Monaco Grand Prix,2009-05-24,12:00:00,http://en.wikipedia.org/wiki/2009_Monaco_Grand...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
6,7,2009,7,5,Turkish Grand Prix,2009-06-07,12:00:00,http://en.wikipedia.org/wiki/2009_Turkish_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
7,8,2009,8,9,British Grand Prix,2009-06-21,12:00:00,http://en.wikipedia.org/wiki/2009_British_Gran...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
8,9,2009,9,20,German Grand Prix,2009-07-12,12:00:00,http://en.wikipedia.org/wiki/2009_German_Grand...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
9,10,2009,10,11,Hungarian Grand Prix,2009-07-26,12:00:00,http://en.wikipedia.org/wiki/2009_Hungarian_Gr...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N


In [81]:
df_races.tail(10)

,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
1115,1135,2024,15,39,Dutch Grand Prix,2024-08-25,13:00:00,https://en.wikipedia.org/wiki/2024_Dutch_Grand...,2024-08-23,10:30:00,2024-08-23,14:00:00,2024-08-24,09:30:00,2024-08-24,13:00:00,\N,\N
1116,1136,2024,16,14,Italian Grand Prix,2024-09-01,13:00:00,https://en.wikipedia.org/wiki/2024_Italian_Gra...,2024-08-30,11:30:00,2024-08-30,15:00:00,2024-08-31,10:30:00,2024-08-31,14:00:00,\N,\N
1117,1137,2024,17,73,Azerbaijan Grand Prix,2024-09-15,11:00:00,https://en.wikipedia.org/wiki/2024_Azerbaijan_...,2024-09-13,09:30:00,2024-09-13,13:00:00,2024-09-14,08:30:00,2024-09-14,12:00:00,\N,\N
1118,1138,2024,18,15,Singapore Grand Prix,2024-09-22,12:00:00,https://en.wikipedia.org/wiki/2024_Singapore_G...,2024-09-20,09:30:00,2024-09-20,13:00:00,2024-09-21,09:30:00,2024-09-21,13:00:00,\N,\N
1119,1139,2024,19,69,United States Grand Prix,2024-10-20,19:00:00,https://en.wikipedia.org/wiki/2024_United_Stat...,2024-10-18,17:30:00,2024-10-18,21:30:00,\N,\N,2024-10-19,22:00:00,2024-10-19,18:00:00
1120,1140,2024,20,32,Mexico City Grand Prix,2024-10-27,20:00:00,https://en.wikipedia.org/wiki/2024_Mexico_City...,2024-10-25,18:30:00,2024-10-25,22:00:00,2024-10-26,17:30:00,2024-10-26,21:00:00,\N,\N
1121,1141,2024,21,18,São Paulo Grand Prix,2024-11-03,17:00:00,https://en.wikipedia.org/wiki/2024_S%C3%A3o_Pa...,2024-11-01,14:30:00,2024-11-01,18:30:00,\N,\N,2024-11-02,18:00:00,2024-11-02,14:00:00
1122,1142,2024,22,80,Las Vegas Grand Prix,2024-11-23,06:00:00,https://en.wikipedia.org/wiki/2024_Las_Vegas_G...,2024-11-21,02:30:00,2024-11-21,06:00:00,2024-11-22,02:30:00,2024-11-22,06:00:00,\N,\N
1123,1143,2024,23,78,Qatar Grand Prix,2024-12-01,17:00:00,https://en.wikipedia.org/wiki/2024_Qatar_Grand...,2024-11-29,13:30:00,2024-11-29,17:30:00,\N,\N,2024-11-30,17:00:00,2024-11-30,13:00:00
1124,1144,2024,24,24,Abu Dhabi Grand Prix,2024-12-08,13:00:00,https://en.wikipedia.org/wiki/2024_Abu_Dhabi_G...,2024-12-06,09:30:00,2024-12-06,13:00:00,2024-12-07,10:30:00,2024-12-07,14:00:00,\N,\N


In [82]:
df_races.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1125 entries, 0 to 1124
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   raceId       1125 non-null   int64 
 1   year         1125 non-null   int64 
 2   round        1125 non-null   int64 
 3   circuitId    1125 non-null   int64 
 4   name         1125 non-null   object
 5   date         1125 non-null   object
 6   time         1125 non-null   object
 7   url          1125 non-null   object
 8   fp1_date     1125 non-null   object
 9   fp1_time     1125 non-null   object
 10  fp2_date     1125 non-null   object
 11  fp2_time     1125 non-null   object
 12  fp3_date     1125 non-null   object
 13  fp3_time     1125 non-null   object
 14  quali_date   1125 non-null   object
 15  quali_time   1125 non-null   object
 16  sprint_date  1125 non-null   object
 17  sprint_time  1125 non-null   object
dtypes: int64(4), object(14)
memory usage: 158.3+ KB


In [83]:
df_races.dtypes

,0
raceId,int64
year,int64
round,int64
circuitId,int64
name,object
date,object
time,object
url,object
fp1_date,object
fp1_time,object


In [85]:
# Checking for '\N' counts in the df_drivers
n_string_counts_races = (df_races == r'\N').sum()
n_string_counts_races

,0
raceId,0
year,0
round,0
circuitId,0
name,0
date,0
time,731
url,0
fp1_date,1035
fp1_time,1057


# Initial Data Audit & Discovery

1.   **Results Table(results.csv)**
*   **Status:** High-volume "dirty" data detected.
*   **Key Findings:** Found 10,953 null strings in position and over 18,000 in rank.
*   **Impact:** This table requires the most aggressive cleaning as it contains the core variables for the "Grid vs. Podium" and "Midfield King" analyses.

2.   **Drivers Table (drivers.csv)**
*   **Status:** Moderate "dirty" data in administrative fields.    
*   **Key Findings** 802 entries in the number column and 757 in the code column contain the \N string.
*   **Obsercation:** These missing values typically occur for historical drivers from eras where permanent racing numbers and 3-letter timing codes (e.g., HAM, VER) were not standardized.

1.   **Races Table (races.csv)**
*   **Status:** High-volume missing telemetry and schedule data.
*   **Key Findings:** The time column has 731 null strings. More significantly, the Free Practice (fp1-3), Qualifying, and Sprint session columns are almost entirely populated with \N (ranging from 1,035 to 1,110 entries).
*   **Impact:** This indicates that session-specific start times were only consistently recorded in the database for very recent seasons.


**Next Discovery Step:** To complete the statistical overview of this stage, we should now run .describe() on all three tables. This will show us the mean, min, and max for years, grid positions, and points.

# Statistical Discovery

In [86]:
#Running descriptive statistics on df_results
df_results.describe()

,resultId,raceId,driverId,constructorId,grid,positionOrder,points,laps,statusId
count,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000,26759.000000
mean,13380.977391,551.687283,278.673530,50.180537,11.134796,12.794051,1.987632,46.301768,17.224971
std,7726.134642,313.265036,282.703039,61.551498,7.202860,7.665951,4.351209,29.496557,26.026104
min,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000
25%,6690.500000,300.000000,57.000000,6.000000,5.000000,6.000000,0.000000,23.000000,1.000000
50%,13380.000000,531.000000,172.000000,25.000000,11.000000,12.000000,0.000000,53.000000,10.000000
75%,20069.500000,811.000000,399.500000,63.000000,17.000000,18.000000,2.000000,66.000000,14.000000
max,26764.000000,1144.000000,862.000000,215.000000,34.000000,39.000000,50.000000,200.000000,141.000000


In [87]:
#Running descriptive statistics on df_drivers
df_drivers.describe()

,driverId
count,861.000000
mean,431.061556
std,248.793797
min,1.000000
25%,216.000000
50%,431.000000
75%,646.000000
max,862.000000


In [88]:
#Running descriptive statistics on df_results
df_races.describe()

,raceId,year,round,circuitId
count,1125.000000,1125.000000,1125.000000,1125.000000
mean,565.710222,1992.703111,8.579556,23.889778
std,328.813817,20.603848,5.159910,19.633527
min,1.000000,1950.000000,1.000000,1.000000
25%,282.000000,1977.000000,4.000000,9.000000
50%,563.000000,1994.000000,8.000000,18.000000
75%,845.000000,2011.000000,13.000000,34.000000
max,1144.000000,2024.000000,24.000000,80.000000


# Statistical Discovery & Descriptive Analysis

1.   **Dataset Scale and Population**

*   **Results Record:** We are analyzing 26,759 individual race results.
*   **Athlete Count:** The dataset spans 861 unique drivers throughout F1 history.
*   **Temporal Range:** The races table confirms our data starts at the very first championship in 1950 and extends to the 2024 season.

2.   **Critical Statistical Insights**

*   **Grid Position (The Start):** The average starting position (grid) is 11.13, but the maximum is 34, indicating older races had much larger starting grids than the modern 20-car standard.
*   **Finishing Position (positionOrder):** The average finishing order is 12.79. The discrepancy between the mean grid (11.1) and mean positionOrder (12.7) highlights the impact of retirements and DNFs across the dataset.
*   **Points Distribution:** The mean points awarded per entry is only 1.98, with a 75th percentile of just 2.0 points. This confirms that for the majority of F1 history, points were a rare commodity reserved for the top finishers, which will be a key factor in our "Midfield King" analysis.

3.   **Identified Anomalies for Stage 3 (Cleaning)**

*   **Zero-Value Laps:** We have entries showing a minimum of 0 laps completed. These represent "DNS" (Did Not Start) or lap-one retirements that must be handled so they
*   **The Grid 0 Mystery:** The minimum value for grid is 0. In F1, grid positions start at 1. A "0" likely indicates a pit-lane start or a disqualification, which we need to investigate during the cleaning phase.
*   **The 200-Lap Outlier:** Descriptive statistics show a maximum of 200 laps. This corresponds to the Indianapolis 500 races included in the championship during the 1950s, a significant outlier compared to modern ~50-70 lap Grands Prix.



# Plan of Action
The discovery of missing schedule data in the races table means we should focus our Exploratory Data Analysis (EDA) on race-day outcomes rather than session start times.

**Revising Handling Strategy**
1.   **Uniform Nullification:** We will use a global approach to replace \N across all dataframes to prevent calculation errors.
2.   **Feature Selection:** For the "Midfield King" analysis, we will prioritize positionOrder (which is clean) over position (which has 10k+ nulls) to ensure we can account for every driver who started the race.
3.   **Dimensional Integrity:** When we join the drivers table to the results table, we must be aware that older drivers will have NaN for their racing numbers.
4.  **Type Casting:** Convert telemetry features from object to float64 or int64 to enable mathematical modeling.